# Predictive Modeling

This notebook develops and evaluates supervised machine learning models for classifying mental health-related text. The cleaned and preprocessed text is transformed into TF-IDF feature vectors using the vectorizer created during the feature engineering phase. Multiple classification algorithms are trained and evaluated using standard performance metrics to identify the most effective model for predicting mental health categories.

In [178]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

## Load the Processed Dataset

Load the cleaned and preprocessed dataset generated during the text preprocessing phase. This dataset contains the processed text and corresponding class labels that will be used for model training and evaluation.

In [179]:
# Define project paths
project_root = Path("..")
data_path = project_root / "data" / "processed" / "mental_health_text_preprocessed.csv"

# Load dataset
df = pd.read_csv(data_path)

# Convert labels to integers
df["Label"] = df["Label"].astype(int)

# Display basic information
print(f"Dataset shape: {df.shape}")
display(df.head())

Dataset shape: (15774, 8)


,Label,TEXT,character_count,word_count,TEXT_ORIGINAL,TEXT_REPAIRED,TEXT_NORMALIZED,TEXT_PROCESSED
0,0,TIL the movie Starship Troopers was never adap...,89,14,TIL the movie Starship Troopers was never adap...,TIL the movie Starship Troopers was never adap...,TIL the movie Starship Troopers was never adap...,til movie starship troopers never adapt succes...
1,0,What do you call a fat baby?,28,7,What do you call a fat baby?,What do you call a fat baby?,What do you call a fat baby?,fat baby
2,0,Two morons are sitting on a fence. The big one...,78,16,Two morons are sitting on a fence. The big one...,Two morons are sitting on a fence. The big one...,Two morons are sitting on a fence. The big one...,number moron sit fence big number fall not
3,0,I covered all my weapons in glue.,33,7,I covered all my weapons in glue.,I covered all my weapons in glue.,I covered all my weapons in glue.,cover weapon glue
4,0,Joke I made up: Caveman and a bear walk into a...,103,19,Joke I made up: Caveman and a bear walk into a...,Joke I made up: Caveman and a bear walk into a...,Joke I made up: Caveman and a bear walk into a...,joke caveman bear walk bar ba ender say story ...


## Validate the Dataset

In [180]:
# Validate dataset structure
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df[["TEXT_PROCESSED", "Label"]].isna().sum())

print("\nClass distribution:")
print(df["Label"].value_counts())


Columns:
['Label', 'TEXT', 'character_count', 'word_count', 'TEXT_ORIGINAL', 'TEXT_REPAIRED', 'TEXT_NORMALIZED', 'TEXT_PROCESSED']

Missing values:
TEXT_PROCESSED    0
Label             0
dtype: int64

Class distribution:
Label
0    6032
2    5446
1    4296
Name: count, dtype: int64


## Load the Saved TF-IDF Vectorizer

Load the TF-IDF vectorizer created during the feature engineering phase. Reusing the fitted vectorizer ensures that the predictive models use the same vocabulary and feature structure established from the training data.

In [181]:
# Define the vectorizer path
models_dir = project_root / "models"
vectorizer_path = models_dir / "tfidf_vectorizer.joblib"

# Load the fitted TF-IDF vectorizer
tfidf = joblib.load(vectorizer_path)

print(f"Vectorizer loaded from: {vectorizer_path}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")

Vectorizer loaded from: ..\models\tfidf_vectorizer.joblib
Vocabulary size: 23,837


## Prepare Features and Labels

Define the processed text as the input feature and the mental health category as the target label. These variables will be divided into training and testing subsets before applying the saved TF-IDF vectorizer.

In [182]:
# Define the input text and target labels
X = df["TEXT_PROCESSED"]
y = df["Label"]

print(f"Number of text records: {len(X):,}")
print(f"Number of labels: {len(y):,}")

Number of text records: 15,774
Number of labels: 15,774


## Split the Data into Training and Testing Sets

Divide the dataset into training and testing subsets using an 80/20 split. Stratified sampling preserves the proportion of each class in both subsets, while the fixed random state makes the split reproducible.

In [183]:
# Split the data while preserving the class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training samples: 12,619
Testing samples: 3,155

Training class distribution:
Label
0    4825
2    4357
1    3437
Name: count, dtype: int64

Testing class distribution:
Label
0    1207
2    1089
1     859
Name: count, dtype: int64


## Transform the Text Using TF-IDF

Transform the training and testing text into numerical feature matrices using the TF-IDF vectorizer loaded from the feature engineering phase. The existing vectorizer is applied without refitting so that the same vocabulary and feature representation are used during model development.

In [184]:
# Transform the training and testing text
X_train_tfidf = tfidf.transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Training feature matrix shape: {X_train_tfidf.shape}")
print(f"Testing feature matrix shape: {X_test_tfidf.shape}")

Training feature matrix shape: (12619, 23837)
Testing feature matrix shape: (3155, 23837)


## Logistic Regression

Train a Logistic Regression classifier using the TF-IDF training features. Logistic Regression is commonly used as a baseline for text classification because it performs efficiently with high-dimensional sparse data.

In [185]:
# Create and train the Logistic Regression model
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

logistic_model.fit(X_train_tfidf, y_train)

# Generate predictions for the testing data
logistic_predictions = logistic_model.predict(X_test_tfidf)

print("Logistic Regression training complete.")

Logistic Regression training complete.


### Logistic Regression Evaluation

Evaluate the model using accuracy and a classification report. The classification report provides precision, recall, and F1-score for each mental health category.

In [186]:
# Evaluate Logistic Regression
logistic_accuracy = accuracy_score(y_test, logistic_predictions)

print(f"Logistic Regression Accuracy: {logistic_accuracy:.4f}")
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        logistic_predictions,
        target_names=["Neutral", "Depression", "Suicidal"],
    )
)

Logistic Regression Accuracy: 0.8840

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.85      0.94      0.90      1207
  Depression       0.93      0.88      0.91       859
    Suicidal       0.89      0.82      0.85      1089

    accuracy                           0.88      3155
   macro avg       0.89      0.88      0.88      3155
weighted avg       0.89      0.88      0.88      3155



In [187]:
# Store Logistic Regression results
results = []

results.append(
    {
        "Model": "Logistic Regression",
        "Accuracy": logistic_accuracy,
        "Precision": precision_score(
            y_test,
            logistic_predictions,
            average="weighted",
        ),
        "Recall": recall_score(
            y_test,
            logistic_predictions,
            average="weighted",
        ),
        "F1-Score": f1_score(
            y_test,
            logistic_predictions,
            average="weighted",
        ),
    }
)

## Multinomial Naïve Bayes

Train a Multinomial Naïve Bayes classifier using the TF-IDF feature matrices. Naïve Bayes is a probabilistic algorithm that has been widely used for text classification because it performs efficiently with word frequency and TF-IDF features.

In [188]:
# Create and train the Multinomial Naïve Bayes model
naive_bayes_model = MultinomialNB()

naive_bayes_model.fit(X_train_tfidf, y_train)

# Generate predictions
naive_bayes_predictions = naive_bayes_model.predict(X_test_tfidf)

print("Multinomial Naïve Bayes training complete.")

Multinomial Naïve Bayes training complete.


In [189]:
# Evaluate Multinomial Naïve Bayes
naive_bayes_accuracy = accuracy_score(y_test, naive_bayes_predictions)

print(f"Multinomial Naïve Bayes Accuracy: {naive_bayes_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        naive_bayes_predictions,
        target_names=[
            "Neutral",
            "Depression",
            "Suicidal",
        ],
    )
)

# Store results
results.append(
    {
        "Model": "Multinomial Naïve Bayes",
        "Accuracy": naive_bayes_accuracy,
        "Precision": precision_score(
            y_test,
            naive_bayes_predictions,
            average="weighted",
        ),
        "Recall": recall_score(
            y_test,
            naive_bayes_predictions,
            average="weighted",
        ),
        "F1-Score": f1_score(
            y_test,
            naive_bayes_predictions,
            average="weighted",
        ),
    }
)

Multinomial Naïve Bayes Accuracy: 0.8162

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.98      0.63      0.77      1207
  Depression       0.84      0.91      0.87       859
    Suicidal       0.71      0.95      0.82      1089

    accuracy                           0.82      3155
   macro avg       0.84      0.83      0.82      3155
weighted avg       0.85      0.82      0.81      3155



## Linear Support Vector Machine

Train a Linear Support Vector Machine (Linear SVM) classifier using the TF-IDF feature matrices. Linear SVM is well suited for high-dimensional text classification problems and is commonly used because of its strong performance with sparse feature data.

In [190]:
# Create and train the Linear SVM model
svm_model = LinearSVC(random_state=42)

svm_model.fit(X_train_tfidf, y_train)

# Generate predictions
svm_predictions = svm_model.predict(X_test_tfidf)

print("Linear SVM training complete.")

Linear SVM training complete.


In [191]:
# Evaluate Linear SVM
svm_accuracy = accuracy_score(
    y_test,
    svm_predictions,
)

print(f"Linear SVM Accuracy: {svm_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        svm_predictions,
        target_names=[
            "Neutral",
            "Depression",
            "Suicidal",
        ],
    )
)

# Store results
results.append(
    {
        "Model": "Linear SVM",
        "Accuracy": svm_accuracy,
        "Precision": precision_score(
            y_test,
            svm_predictions,
            average="weighted",
        ),
        "Recall": recall_score(
            y_test,
            svm_predictions,
            average="weighted",
        ),
        "F1-Score": f1_score(
            y_test,
            svm_predictions,
            average="weighted",
        ),
    }
)

Linear SVM Accuracy: 0.8840

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.88      0.92      0.90      1207
  Depression       0.92      0.88      0.90       859
    Suicidal       0.86      0.84      0.85      1089

    accuracy                           0.88      3155
   macro avg       0.89      0.88      0.88      3155
weighted avg       0.88      0.88      0.88      3155



## Random Forest

Train a Random Forest classifier using the TF-IDF feature matrices. Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve prediction accuracy and reduce overfitting. It is included to compare a tree-based approach with the linear and probabilistic models.

In [192]:
# Create and train the Random Forest model
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)

random_forest_model.fit(X_train_tfidf, y_train)

# Generate predictions
random_forest_predictions = random_forest_model.predict(X_test_tfidf)

print("Random Forest training complete.")

Random Forest training complete.


In [193]:
# Evaluate Random Forest
random_forest_accuracy = accuracy_score(
    y_test,
    random_forest_predictions,
)

print(f"Random Forest Accuracy: {random_forest_accuracy:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        random_forest_predictions,
        target_names=[
            "Neutral",
            "Depression",
            "Suicidal",
        ],
    )
)

# Store results
results.append(
    {
        "Model": "Random Forest",
        "Accuracy": random_forest_accuracy,
        "Precision": precision_score(
            y_test,
            random_forest_predictions,
            average="weighted",
        ),
        "Recall": recall_score(
            y_test,
            random_forest_predictions,
            average="weighted",
        ),
        "F1-Score": f1_score(
            y_test,
            random_forest_predictions,
            average="weighted",
        ),
    }
)

Random Forest Accuracy: 0.8621

Classification Report:
              precision    recall  f1-score   support

     Neutral       0.85      0.91      0.88      1207
  Depression       0.92      0.85      0.89       859
    Suicidal       0.84      0.81      0.82      1089

    accuracy                           0.86      3155
   macro avg       0.87      0.86      0.86      3155
weighted avg       0.86      0.86      0.86      3155



## Model Performance Comparison

Compare the performance of the machine learning models using accuracy and weighted precision, recall, and F1-score. This summary supports the selection of the strongest-performing model for the mental health text classification task.

In [194]:
# Create and sort the model comparison table
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["Accuracy", "F1-Score"],
    ascending=False,
).reset_index(drop=True)

# Format metrics for easier comparison
display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1-Score
0,Linear SVM,0.8840,0.8843,0.8840,0.8838
1,Logistic Regression,0.8840,0.8858,0.8840,0.8834
2,Random Forest,0.8621,0.8636,0.8621,0.8620
3,Multinomial Naïve Bayes,0.8162,0.8503,0.8162,0.8123


## Save the Selected Model

Logistic Regression and Linear SVM achieved the same overall accuracy. Logistic Regression was selected as the final model because it provides comparable performance while also supporting probability estimates and straightforward interpretation. The fitted model is saved for future use.

In [195]:
# Save the selected final model
best_model_path = models_dir / "best_model.joblib"

joblib.dump(logistic_model, best_model_path)

print(f"Selected model saved to: {best_model_path}")

Selected model saved to: ..\models\best_model.joblib


## Summary

Four supervised machine learning models were trained and evaluated for mental health text classification using TF-IDF features. Logistic Regression and Linear Support Vector Machine achieved the highest overall accuracy at 88.4%, followed by Random Forest at 86.2% and Multinomial Naïve Bayes at 81.6%.

Although Logistic Regression and Linear SVM produced the same accuracy, their class-level results differed slightly. Logistic Regression achieved stronger precision for the Suicidal category and a slightly higher F1-score for the Depression category, while Linear SVM produced higher recall for the Suicidal category. Logistic Regression was selected as the final model because it provided strong and balanced classification performance while also supporting probability estimates and straightforward interpretation.